In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Define XOR gate data
# Input: All combinations of two binary variables
# Output: XOR truth table
X = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
y = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

In [2]:
# 2. Define model: Requires at least one hidden layer for XOR (not linearly separable)
class XORGateNN(nn.Module):
    def __init__(self):
        super(XORGateNN, self).__init__()
        # Hidden layer with 2 neurons and ReLU activation
        self.hidden = nn.Linear(2, 2) 
        # Output layer with 1 neuron and sigmoid activation
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        x = torch.relu(self.hidden(x))  # ReLU activation on hidden layer
        return torch.sigmoid(self.output(x))  # Sigmoid activation on output for binary classification


## ⚠️ Why this model works for XOR:

- XOR is **not linearly separable**, so a **single-layer perceptron won't work**.
- This model works because it includes:
  - ✅ A **hidden layer** (adds non-linearity with **ReLU** activation)
  - ✅ A **sigmoid output layer** (for binary prediction)
  - ✅ **Binary Cross Entropy Loss** (suitable for binary classification tasks with 0 or 1 labels)


In [3]:
model = XORGateNN()

# 3. Define loss and optimizer
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for binary classification
optimizer = optim.SGD(model.parameters(), lr=0.1)  # Stochastic Gradient Descent with learning rate 0.1

# 4. Train the model
for epoch in range(10000):  # More epochs since XOR is harder to learn than AND
    optimizer.zero_grad()         # Clear previous gradients
    outputs = model(X)            # Forward pass
    loss = criterion(outputs, y)  # Compute loss
    loss.backward()              # Backward pass
    optimizer.step()             # Update parameters


In [4]:
# 5. Evaluate the model
with torch.no_grad():
    predictions = model(X)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y).float().mean()
    print(f"Loss: {loss.item():.4f}, Accuracy: {accuracy.item()*100:.2f}%")


Loss: 0.4775, Accuracy: 75.00%


In [5]:
# 6. Save the trained model
torch.save(model.state_dict(), 'xor_gate_model.pth')

# 7. Load the saved model
loaded_model = XORGateNN()
loaded_model.load_state_dict(torch.load('xor_gate_model.pth'))
loaded_model.eval()

# 8. Use loaded model for prediction
with torch.no_grad():
    pred = loaded_model(X)
    print("Predictions:", torch.round(pred).view(-1).tolist())

Predictions: [0.0, 0.0, 1.0, 0.0]


C:\Users\user\AppData\Local\Temp\ipykernel_5908\1250979567.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load('xor_gate_model.pth'))